In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.types import *

c_schema = StructType([
    StructField('customer_id',IntegerType()),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('membership',StringType()),
    StructField('start_date',DateType()),
    StructField('end_date',DateType()),
    StructField('is_current',StringType()),
])

s_schema = StructType(
    [
        StructField('store_id',IntegerType()),
        StructField('store_name',StringType()),
        StructField('store_city',StringType()),
        StructField('store_region',StringType()),
        StructField('start_date',DateType()),
        StructField('end_date',DateType()),
        StructField('is_current',StringType()),
    ]
)

o_schema = StructType([
    StructField('order_id',IntegerType()),
    StructField('customer_id',IntegerType()),
    StructField('store_id',IntegerType()),
    StructField('product_name',StringType()),
    StructField('amount',IntegerType()),
    StructField('order_dt',DateType()),
])


store = spark.read.format('csv').option('header','true').schema(s_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoGWithSCD2/store_source.csv')

customers = spark.read.format('csv').option('header','true').schema(c_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoGWithSCD2/customers_source.csv')

orders = spark.read.format('csv').option('header','true').schema(o_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoGWithSCD2/orders_source.csv')

# Clean
customers = customers.dropna(subset=('customer_id'))
customers = customers.dropDuplicates()
# Skipping window filter
customers  = customers.fillna({
    'customer_id':0,
    'customer_name':'Unknown',
    'city':'Unknown',
    'membership':'Unknown'
})

#Standardization '
customers = customers.withColumn('customer_id',f.initcap(f.trim('customer_id'))).withColumn('customer_name',f.initcap(f.trim('customer_name'))).withColumn('city',f.initcap(f.trim('city'))).withColumn('membership',f.initcap(f.trim('membership')))

# Data Validation Skip
# Audit Column
customers = customers.withColumn('current_dt',f.lit(f.current_date())).withColumn('updated_dt',f.lit(None))

# Clean
store = store.dropna(subset=('store_id'))
store = store.dropDuplicates()
# Skipping window filter
store = store.fillna({
    'store_name':'Unknown',
    'store_city':'Unknown',
    'store_region':'Unknown',
    'is_current':'Unknown'
})

#Standardization
store = store.withColumn('store_name',f.initcap(f.trim(f.col('store_name')))).withColumn('store_city',f.initcap(f.trim(f.col('store_city')))).withColumn('store_region',f.initcap(f.trim(f.col('store_region')))).withColumn('is_current',f.initcap(f.trim(f.col('is_current'))))

# Data Validation Skip
# Audit Column
store = store.withColumn('current_dt',f.lit(f.current_date())).withColumn('updated_dt',f.lit(None))

# Clean
orders = orders.dropna(subset=('customer_id'))
orders = orders.dropDuplicates()

orders = orders.fillna({
    'product_name':'Unknown'
})

#Standardization
orders = orders.withColumn('product_name',f.initcap(f.trim(f.col('product_name'))))
orders = orders.filter(f.col('amount')>=0)

# Data Validation Skip
# Audit Column
orders = orders.withColumn('current_dt',f.lit(f.current_date())).withColumn('updated_dt',f.lit(None))

# Incremental Load ----------------------------------------------------------------------------------#

customers_incremental = spark.read.format('csv').option('header','true').schema(c_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoGWithSCD2/customers_incremental.csv')

store_incremental = spark.read.format('csv').option('header','true').schema(s_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoGWithSCD2/store_incremental.csv')

customers_incremental = customers_incremental.dropna(subset=('customer_id'))
customers_incremental = customers_incremental.dropDuplicates()
#skip window filter
customers_incremental = customers_incremental.fillna({
    'customer_name':'Unknown',
    'city':'Unknown',
    'membership':'Unknown'
})
#standarization
customers_incremental = customers_incremental.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('membership',f.initcap(f.trim(f.col('membership'))))
#data validation skip
#audit columns
customers_incremental = customers_incremental.withColumn('current_dt',f.lit(None)).withColumn('updated_dt',f.lit(f.current_date()))

store_incremental = store_incremental.dropna(subset=('store_id'))
store_incremental = store_incremental.dropDuplicates()
#skip window filter
store_incremental = store_incremental.fillna({
    'store_name':'unknown',
    'store_city':'unknown',
    'store_region':'unknown'
})
#standarization
store_incremental = store_incremental.withColumn('store_name',f.initcap(f.trim(f.col('store_name')))).withColumn(
    'store_city',f.initcap(f.trim(f.col('store_city')))
).withColumn('store_region',f.initcap(f.trim(f.col('store_region'))))
#data validation skip
#audit columns
store_incremental = store_incremental.withColumn('current_dt',f.lit(None)).withColumn('updated_dt',f.lit(f.current_date()))


#record changes
store_updated = store_incremental.alias('i').join(store.alias('s'),on='store_id',how='inner').where(
    (f.col('i.store_name') != f.col('s.store_name')) | (f.col('i.store_city') != f.col('s.store_city')) | (f.col('i.store_region') != f.col('s.store_region'))
).select('i.*')

store_new = store_incremental.alias('i').join(store.alias('s'),on='store_id',how='left_anti')

customers_updated = customers_incremental.alias('c1').join(customers.alias('c2'),on='customer_id',how='inner').where(
    (f.col('c1.customer_name') != f.col('c2.customer_name')) | (f.col('c1.city') != f.col('c2.city'))
    | (f.col('c1.membership') != f.col('c2.membership'))
).select('c1.*')

customers_new = customers_incremental.alias('c1').join(customers.alias('c2'),on='customer_id',how='left_anti')
customers_new = customers_new.withColumn('is_current',f.lit('Y'))
# SCD Type 2
#identify
customers = (customers.alias('c').join(customers_updated.alias('cu'),on='customer_id',how='left').withColumn('new_is_current',f.when(f.col('cu.customer_id').isNotNull(),f.lit('N')).otherwise(f.lit('Y')))).select('c.customer_id','c.customer_name','c.city','c.membership','c.start_date','c.end_date',f.col("new_is_current").alias("is_current"),'c.current_dt','c.updated_dt')

customers_updated = customers_updated.withColumn('is_current',f.lit('Y'))

silver_customers = customers.unionByName(customers_updated).unionByName(customers_new)

store = store.alias('c').join(store_updated.alias('u'),on='store_id',how='left').withColumn('new_is_current',
f.when(f.col('u.store_id').isNotNull(),f.lit('N')).otherwise(f.lit('Y'))).select('c.store_id','c.store_name','c.store_city','c.store_region','c.start_date','c.end_date',f.col('new_is_current').alias('is_current'),'c.current_dt','c.updated_dt')

store_updated = store_updated.withColumn('is_current',f.lit('Y'))

silver_store = store.unionByName(store_updated).unionByName(store_new)

silver_customers = silver_customers.filter(f.col('is_current') == "Y").orderBy('customer_id')
silver_store = silver_store.filter(f.col('is_current') == "Y").orderBy('store_id')
# silver_customers.show()
# silver_store.show()
# orders.show()

#lookup table
gold_orders = orders.alias('o').join(silver_customers.alias('c'),on='customer_id',how='left').join(silver_store.alias('s'),on='store_id',how='left').select('o.order_id','o.customer_id','o.store_id','o.product_name','o.amount','c.customer_name','c.city','c.membership','s.store_name','s.store_city','s.store_region','c.is_current','c.current_dt','c.updated_dt'
).orderBy('o.customer_id')

gold_orders.show()

gold_orders.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('gold_orders')

#requirement1
re1 = gold_orders.groupby('customer_id','customer_name').agg(
    f.count('order_id').alias('total_orders'),
    f.sum('amount').alias('total_sales'),
    f.round(f.avg('amount')).alias('avgOrderValue'),
    f.max('amount').alias('highestOrderValue'),
    f.countDistinct(f.col('store_id')).alias('storeVisited')
)

#requiremnet2 
re2 = gold_orders.groupby('customer_id','customer_name').agg(
    f.sum('amount').alias('total_sales'),
    f.count('order_id').alias('total_orders'),
    f.countDistinct('store_id').alias('store_visited')
)

re2 = re2.withColumn(
    'membership',
    f.when((f.col('total_sales') > 150000) & (f.col('total_orders') >= 3) & (f.col('store_visited')>1),'Platinum').when((f.col('total_sales').between(75000,150000)),'Gold')
    .otherwise('Silver')
)

# requirement3
re3 = gold_orders.groupby('customer_id','customer_name').agg(
    f.max('amount').alias('maxAmount'),
    f.round(f.avg('amount')).alias('avgAmount')
).filter(f.col('maxAmount') > f.col('avgAmount'))


# requirement4
re4 = gold_orders.groupby('customer_id','customer_name').agg(
    f.countDistinct('order_id').alias('total_orders'),
    f.sum('amount').alias('total_amount')
).withColumn('riskCheck',f.when((f.col('total_orders')==1) & (f.col('total_amount')<100000),'Risk').otherwise('Safe'))

re4.show()



In [0]:
%sql
select 
customer_id,customer_name,count(distinct order_id) as total_orders,sum(amount) as total_sales,
case when (count(distinct order_id) = 1 and sum(amount) < 100000) then 'Risk'
     else 'Active' end as riskCheck
from gold_orders
group by customer_id,customer_name